In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Create directory
!mkdir -p /content/bobiac_data_cellpose
# Download the data
!wget https://raw.githubusercontent.com/bobiac/bobiac-book/main/_static/data/05_segmentation_cellpose_training.zip -O /content/bobiac_data_cellpose/05_segmentation_cellpose_training.zip
# Unzip the data, remove zip file and macOS metadata files (if any)
!cd /content/bobiac_data_cellpose && unzip 05_segmentation_cellpose_training.zip && rm -f 05_segmentation_cellpose_training.zip && rm -rf __MACOSX

In [ ]:
!pip install cellpose
!pip install matplotlib

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from cellpose import core, io, metrics, models, train

In [ ]:
io.logger_setup()  # to get printing of progress

use_gpu = core.use_gpu()
print("GPU available:", use_gpu)

In [ ]:
ROOT_FOLDER_PATH = Path("bobiac_data_cellpose/05_segmentation_cellpose_training")

train_dir = ROOT_FOLDER_PATH / "train"
test_dir = ROOT_FOLDER_PATH / "test"

# add name filters to select only images and masks from the folders
# `mask_filter` identifies mask files by their suffix
# (e.g. "_seg" for files like "img_000_seg". If not .tif, add also the extension).
mask_filter = "_seg"

# if necessary, you can also specify an `image_filter` to select images with a specific
# suffix (e.g. "_img" for files like "img_000_raw.tif". If not .tif, add also the extension).
# image_filter = "_raw"

# Load training and test data
output = io.load_train_test_data(
    str(train_dir),
    str(test_dir),
    mask_filter=mask_filter,
    # image_filter=image_filter
)

# assign the output to the appropriate variables
train_data, train_labels, _, test_data, test_labels, _ = output

In [1]:
from cellpose.models import MODEL_DIR
from cellpose.utils import download_url_to_file

model_name = "cpsam_v2"  # or "cpdino" / "cpdino-vitb"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODEL_DIR / model_name
if not model_path.exists():
    url = f"https://huggingface.co/mouseland/cellpose-sam/resolve/main/{model_name}"
    download_url_to_file(url, str(model_path))

  0%|          | 0.00/1.15G [00:00<?, ?B/s]

  0%|          | 8.00k/1.15G [00:00<26:00:03, 13.2kB/s]

  1%|          | 11.9M/1.15G [00:00<00:52, 23.1MB/s]   

  2%|▏         | 21.3M/1.15G [00:00<00:31, 38.9MB/s]

  2%|▏         | 29.2M/1.15G [00:00<00:25, 47.6MB/s]

...

 90%|█████████ | 1.03G/1.15G [00:11<00:00, 188MB/s]

 93%|█████████▎| 1.07G/1.15G [00:11<00:00, 225MB/s]

 96%|█████████▌| 1.10G/1.15G [00:11<00:00, 257MB/s]

 98%|█████████▊| 1.13G/1.15G [00:11<00:00, 195MB/s]

100%|██████████| 1.15G/1.15G [00:11<00:00, 105MB/s]

In [ ]:
model_path = str(MODEL_DIR / "cpsam_v2")  # or "cpdino" / "cpdino-vitb" or "cpsam"
model = models.CellposeModel(pretrained_model=model_path, gpu=use_gpu)

In [ ]:
# run model on test images
masks, _, _ = model.eval(test_data, batch_size=8)

In [ ]:
# check performance using ground truth labels
# average_precision returns AP at IoU thresholds [0.5, 0.75, 0.9] by default
values = metrics.average_precision(test_labels, masks)
average_precision, _, _, _ = values

print(f"average precision at iou threshold 0.5  = {average_precision[:, 0].mean():.3f}")
print(f"average precision at iou threshold 0.75 = {average_precision[:, 1].mean():.3f}")
print(f"average precision at iou threshold 0.9  = {average_precision[:, 2].mean():.3f}")

In [ ]:
n = 0  # test image index to visualize
cyto_ch = 1  # channel index for cytoplasm (0=nucleus, 1=cytoplasm in this dataset)
raw_data = test_data[n][cyto_ch]  # selecting which test data ans which channel
pred_mask = masks[n]  # selecting the predicted mask for the same test image
gt_mask = test_labels[n]  # selecting the ground truth mask for the same test image

plt.figure(figsize=(10, 5))
plt.subplot(1, 3, 1)

plt.imshow(raw_data, cmap="gray")
plt.title(f"Test Image {n}")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(pred_mask, cmap="nipy_spectral")
plt.title(f"Predicted Mask {n}")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(gt_mask, cmap="nipy_spectral")
plt.title(f"GT Mask {n}")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# path and name for saving the trained model
save_path = ROOT_FOLDER_PATH
model_name = "new_model"

# Training params - here we only change the number of epochs and images per epoch
# but you can change other parameters as well, see the dropdown above or the Cellpose\
# API documentation for details.

n_epochs = 10  # using 10 to speed up the training for this tutorial
nimg_per_epoch = 5  # using 5 to speed up the training for this tutorial

new_model_path, train_losses, test_losses = train.train_seg(
    model.net,
    train_data=train_data,
    train_labels=train_labels,
    test_data=test_data,
    test_labels=test_labels,
    n_epochs=n_epochs,
    nimg_per_epoch=nimg_per_epoch,
    model_name=model_name,
    save_path=save_path,
    load_files=False,  # we already loaded the data above with `io.load_train_test_data`
)

# NOTE: to speed up the training you can omit the test data and test labels from the
# `train_seg` function, but then you won't get test losses or a model saved at the epoch
# with the best test loss.

In [ ]:
fig, ax = plt.subplots()
ax.plot(train_losses, label="train loss")
ax.plot(test_losses, label="test loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training and Test Losses")
ax.legend()
plt.show()

In [ ]:
# load the newly trained model
model = models.CellposeModel(pretrained_model=new_model_path, gpu=use_gpu)

# run model on test images
masks, _, _ = model.eval(test_data, batch_size=8)

# check performance using ground truth labels
# average_precision returns AP at IoU thresholds [0.5, 0.75, 0.9] by default
values = metrics.average_precision(test_labels, masks)
average_precision, _, _, _ = values

print(f"average precision at iou threshold 0.5  = {average_precision[:, 0].mean():.3f}")
print(f"average precision at iou threshold 0.75 = {average_precision[:, 1].mean():.3f}")
print(f"average precision at iou threshold 0.9  = {average_precision[:, 2].mean():.3f}")